# Build `questions_no_hint.csv`

Notebook per trasformare `data/evaluation/questions.csv` in un CSV senza opzioni multiple-choice e con risposta testuale corretta.


## 0) Setup percorsi e import


In [ ]:
from pathlib import Path
import re
import sys

import pandas as pd


def _find_repo_root_for_bootstrap(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'notebooks' / 'pipelines' / 'common' / 'bootstrap.py').exists() and (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Project root non trovato. Avvia il notebook dentro il repository.')


_BOOTSTRAP_ROOT = _find_repo_root_for_bootstrap(Path.cwd().resolve())
if str(_BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(_BOOTSTRAP_ROOT))

from notebooks.pipelines.common.bootstrap import bootstrap_notebook

ROOT, _ = bootstrap_notebook(start=Path.cwd().resolve(), load_env=False)
INPUT_CSV = ROOT / 'data' / 'evaluation' / 'questions.csv'
OUTPUT_CSV = ROOT / 'data' / 'evaluation' / 'questions_no_hint.csv'

print('Input:', INPUT_CSV)
print('Output:', OUTPUT_CSV)
print('Input exists:', INPUT_CSV.exists())


## 1) Load CSV originale


In [ ]:
df = pd.read_csv(INPUT_CSV, encoding="utf-8")
print("Shape originale:", df.shape)
df.head(3)


## 2) Regex helpers


In [ ]:
OPTION_RE = re.compile(r"(?m)^\s*([A-F])\)\s+(.*)$") # Opzioni etichettate da A) a F), con testo dopo la parentesi. Multilinea per catturare tutte le opzioni.
EXPECTED_LABELS = ["A", "B", "C", "D", "E", "F"]

def _safe_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value)

def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def extract_stem_and_options(domanda: str) -> tuple[str, dict[str, str]]:
    domanda = _safe_text(domanda).strip("\n")
    matches = list(OPTION_RE.finditer(domanda))
    if not matches:
        raise ValueError("No options found in `Domanda` field")

    stem = normalize_whitespace(domanda[: matches[0].start()])
    options = {m.group(1).strip(): normalize_whitespace(m.group(2)) for m in matches}

    labels = list(options.keys())
    if labels != EXPECTED_LABELS:
        raise ValueError(f"Unexpected option labels/order: {labels!r}")

    return stem, options

def normalize_references(reference_text: str) -> str:
    parts = [
        normalize_whitespace(line)
        for line in _safe_text(reference_text).splitlines()
        if line and line.strip()
    ]
    return " | ".join(parts)

def map_answer(row: pd.Series) -> dict[str, str]:
    qid = normalize_whitespace(_safe_text(row.get("#", ""))) or "<missing>"

    stem, options = extract_stem_and_options(_safe_text(row.get("Domanda", "")))

    answer_label = normalize_whitespace(_safe_text(row.get("Risposta corretta", ""))).upper()
    if answer_label not in options:
        raise ValueError(f"Q{qid}: answer label {answer_label!r} not found in options")

    return {
        "Domanda": stem,
        "Livello": normalize_whitespace(_safe_text(row.get("Livello", ""))),
        "Risposta corretta": options[answer_label],
        "Riferimento legge per la risposta": normalize_references(row.get("Riferimento legge per la risposta", "")),
    }


## 3) Transform


In [ ]:
# Escludo righe vuote finali e qualunque riga senza risposta corretta.
mask_valid = df["Risposta corretta"].notna() & (df["Risposta corretta"].astype(str).str.strip() != "")
df_valid = df.loc[mask_valid].copy()

print("Righe valide:", len(df_valid))

records = [map_answer(row) for _, row in df_valid.iterrows()]
df_out = pd.DataFrame(
    records,
    columns=["Domanda", "Livello", "Risposta corretta", "Riferimento legge per la risposta"],
)

print("Shape output:", df_out.shape)
df_out.head(3)


## 4) Validation


In [ ]:
assert list(df_out.columns) == [
    "Domanda",
    "Livello",
    "Risposta corretta",
    "Riferimento legge per la risposta",
]

assert not df_out.isna().any().any(), "Sono presenti NaN nell'output"
assert len(df_out) == 100, f"Numero righe inatteso: {len(df_out)}"

expected_levels = {"L1": 25, "L2": 25, "L3": 25, "L4": 25}
actual_levels = df_out["Livello"].value_counts().sort_index().to_dict()
assert actual_levels == expected_levels, f"Distribuzione livelli inattesa: {actual_levels}"

print("Distribuzione livelli:", actual_levels)

sample_indices = [0, len(df_out) // 2, len(df_out) - 1]
df_out.iloc[sample_indices]


## 5) Write output CSV


In [ ]:
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("File scritto:", OUTPUT_CSV)
print("Esiste:", OUTPUT_CSV.exists())


## 6) Preview output


In [ ]:
df_preview = pd.read_csv(OUTPUT_CSV, encoding="utf-8")
print("Shape riletta:", df_preview.shape)
df_preview.head(10)
